#ML2 Laboratorio 2 (PDDL+Fastdownward)
##Recuperación de servicio en un centro de datos — Solucion docente

**Qué hace este notebook**:
- Genera `domain_datacenter_lab2.pddl` y `problem_datacenter_lab2.pddl`.
- Carga el problema con **Unified Planning**.
- Ejecuta el motor **Fast Downward** (vía `up-fast-downward`).
- Muestra el plan y guarda una copia en `outputs/plan.txt`.

> Nota: Este lab está diseñado para practicar **modelado PDDL** (UD1) y empezar a experimentar con **Fast Downward**(UD2).

In [ ]:
#Precondicion GOOGLE COLAB: instala dependencias al runtime actual.
#Si trabajas en local/conda y ya lo tienes instalado, puedes saltarte esta celda.
!pip -q install unified-planning up-fast-downward

import unified_planning as up
print('Unified Planning instalado. Version:', getattr(up, '__version__', 'unknown'))


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 743.5/743.5 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.6/117.6 kB 7.1 MB/s eta 0:00:00
Unified Planning instalado. Version: 1.3.0


##1. Ideas clave del diseño del dominio
Modelamos un entorno de planificación clásica:

Este dominio representa recuperación de servicios:
- Un servicio `svc` debe estar **desplegado/deployado** en un servidor `s`.
- Para poder restaurar/arrancar, el servidor debe estar **encendido** y **no aislado**.
- La red necesaria debe estar **operativa**.
- Además, `svcA` depende de `svcB` (no se puede arrancar A sin B arrancado).

Manteniendo planificación clásica (sin tiempo/costes), usamos STRIPS puro con precondiciones y efectos.


In [ ]:
from pathlib import Path

#Definimos el dominio
domain_text = '''(define (domain datacenter_recovery)
  (:requirements :strips :typing)
  (:types servidor servicio red snapshot)

  (:predicates
    (encendido ?s - servidor)
    (aislado ?s - servidor)

    (red_ok ?n - red)

    (deployado ?svc - servicio ?s - servidor)
    (restaurado ?svc - servicio)
    (arrancado ?svc - servicio)

    (dependencia ?svc - servicio ?dep - servicio)
    (snapshot_disponible ?snap - snapshot ?svc - servicio)
  )

  (:action encender_servidor
    :parameters (?s - servidor)
    :precondition (and (not (encendido ?s)))
    :effect (and (encendido ?s))
  )

  (:action desaislar_servidor
    :parameters (?s - servidor)
    :precondition (and (aislado ?s))
    :effect (and (not (aislado ?s)))
  )

  (:action habilitar_red
    :parameters (?n - red)
    :precondition (and (not (red_ok ?n)))
    :effect (and (red_ok ?n))
  )

  (:action restaurar_servicio
    :parameters (?svc - servicio ?s - servidor ?n - red ?snap - snapshot)
    :precondition (and
      (encendido ?s)
      (not (aislado ?s))
      (red_ok ?n)
      (deployado ?svc ?s)
      (snapshot_disponible ?snap ?svc)
      (not (restaurado ?svc))
    )
    :effect (and (restaurado ?svc))
  )

  (:action arrancar_servicio
    :parameters (?svc - servicio ?s - servidor ?n - red ?dep - servicio)
    :precondition (and
      (encendido ?s)
      (not (aislado ?s))
      (red_ok ?n)
      (deployado ?svc ?s)
      (restaurado ?svc)

      ;; Dependencia explícita: el servicio ?svc depende de ?dep y ?dep debe estar arrancado
      (dependencia ?svc ?dep)
      (arrancado ?dep)

      (not (arrancado ?svc))
    )
    :effect (and (arrancado ?svc))
  )

  ;;Para permitir arrancar servicios sin dependencias, añadimos una accion alternativa
  ;;os reocmiendo anadir todas las acciones que podais para darle mas versatilidad al problema
  (:action arrancar_sin_dependencias
    :parameters (?svc - servicio ?s - servidor ?n - red)
    :precondition (and
      (encendido ?s)
      (not (aislado ?s))
      (red_ok ?n)
      (deployado ?svc ?s)
      (restaurado ?svc)
      (not (arrancado ?svc))
    )
    :effect (and (arrancado ?svc))
  )
)
'''

#definimos el problema
problem_text = '''(define (problem datacenter_recovery_p2)
  (:domain datacenter_recovery)
  (:requirements :strips :typing)

  (:objects
    s1 s2 s3 s4 - servidor
    svcA svcB svcC - servicio
    net1 net2 - red
    snap1 snap2 snap3 - snapshot
  )

  (:init
    ;; Redes: net2 caída inicialmente
    (red_ok net1)
    ;; (red_ok net2)  ; net2 NO está ok

    ;; Servidores: s2 apagado, s3 aislado
    (encendido s1)
    (encendido s3)
    (encendido s4)
    (aislado s3)

    ;; Deployments
    (deployado svcA s3)
    (deployado svcB s2)
    (deployado svcC s1)

    ;; Dependencia: A depende de B
    (dependencia svcA svcB)

    ;; Snapshots disponibles (ahora el problema ES resoluble)
    (snapshot_disponible snap1 svcB)
    (snapshot_disponible snap2 svcC)
    (snapshot_disponible snap3 svcA)
  )

  ;; Objetivo: servicio A arrancado (implica arrancar B antes)
  (:goal (and (arrancado svcA)))
)
'''

pddl_dir = Path('pddl_lab2')
pddl_dir.mkdir(exist_ok=True)
domain_path = pddl_dir / 'domain_datacenter_lab2.pddl'
problem_path = pddl_dir / 'problem_datacenter_lab2.pddl'

domain_path.write_text(domain_text, encoding='utf-8')
problem_path.write_text(problem_text, encoding='utf-8')

print('Archivos PDDL generados:')
print(domain_path.resolve())
print(problem_path.resolve())


Archivos PDDL generados:
/content/pddl_lab2/domain_datacenter_lab2.pddl
/content/pddl_lab2/problem_datacenter_lab2.pddl


##2. Smoke test: comprobar que Fast Downward está disponible y puede generar el plan

Usamos OneShotPlanner como el planificador más rápido y simple de UP.

Esta celda intenta resolver **este** problema (no uno dummy). Si falla, suele ser por:
- no tener `up-fast-downward` instalado en el kernel actual,
- o porque el entorno del notebook no coincide con el entorno donde instalaste el paquete.


In [ ]:
#Ahora hacemos los imports, abandonamos el mundo de PDDL para utilizar UP
import unified_planning as up
from unified_planning.io import PDDLReader
from unified_planning.shortcuts import OneshotPlanner

print('Unified Planning version:', getattr(up, '__version__', 'unknown'))

#Cargamos los PDDL generados
reader = PDDLReader()
problem = reader.parse_problem(domain_path.as_posix(), problem_path.as_posix())
print('Problema cargado:', problem.name)
print(' - #Acciones:', len(problem.actions))
print(' - #Fluents:', len(problem.fluents))
print(' - #Objetos:', len(problem.all_objects))

#Intentamos lanzar el planificador
#utilizad el patron try-catch para asegurarnos que podemos cazar cualquier error que surja
try:
    #utilizamos OneshotPlanner para resolver el problema de planificacion de una sola vez (un problema: un plan)
    with OneshotPlanner(name='fast-downward') as planner:
        result = planner.solve(problem)
    print('Planner status:', result.status)

    #Una vez intentamos generarlo vamos a ver que encuentra
    if result.plan is None:
        print('No se encontró plan. Revisa modelado (init/goal/precondiciones).')
    else:
        print('\nPlan encontrado:')
        for i, a in enumerate(result.plan.actions, 1):
            print(f'{i}. {a}')
except Exception as e:
    print('\n No se pudo usar Fast Downward desde Unified Planning.')
    print('Instala en el mismo entorno/kernel: pip install unified-planning up-fast-downward')
    print('Error técnico:', repr(e))

Unified Planning version: 1.3.0
Problema cargado: datacenter_recovery_p2
 - #Acciones: 6
 - #Fluents: 8
 - #Objetos: 12
NOTE: To disable printing of planning engine credits, add this line to your code: `up.shortcuts.get_environment().credits_stream = None`
  *** Credits ***
  * In operation mode `OneshotPlanner` at line 20 of `/tmp/ipykernel_635/359254240.py`, you are using the following planning engine:
  * Engine name: Fast Downward
  * Developers:  Uni Basel team and contributors (cf. https://github.com/aibasel/downward/blob/main/README.md)
  * Description: Fast Downward is a domain-independent classical planning system.

Planner status: PlanGenerationResultStatus.SOLVED_SATISFICING

Plan encontrado:
1. desaislar_servidor(s3)
2. restaurar_servicio(svca, s3, net1, snap3)
3. arrancar_sin_dependencias(svca, s3, net1)


##3. Guardar el plan (si existe)

Guardamos el plan en un archivo para que se pueda adjuntar como evidencia.

In [ ]:

out_dir = Path('outputs')
out_dir.mkdir(exist_ok=True)
plan_path = out_dir / 'plan.txt'

#Reutilizamos result si existe y contiene plan
if 'result' in globals() and result.plan is not None:
    plan_lines = [str(a) for a in result.plan.actions]
    plan_path.write_text('\n'.join(plan_lines), encoding='utf-8')
    print('Plan guardado en:', plan_path.resolve())
else:
    print('No hay plan para guardar (o la planificación no se ejecutó correctamente).')

Plan guardado en: /content/outputs/plan.txt


##4. Aprender a interpretar el plan

Guardamos el plan en un archivo para que se pueda adjuntar como evidencia.
Un plan típico debería incluir pasos como:
1) `encender_servidor s2` (porque `svcB` está en s2 y s2 empieza apagado)
2) `desaislar_servidor s3` (porque `svcA` está en s3 y s3 empieza aislado)
3) `restaurar_servicio svcB s2 net1 snap1`
4) `arrancar_sin_dependencias svcB s2 net1`
5) `restaurar_servicio svcA s3 net1 snap3`
6) `arrancar_servicio svcA s3 net1 svcB`

### Nota de Mar :)
Si no te aparece un plan, normalmente es por un error de modelado (por ejemplo, falta un snapshot, una precondición demasiado fuerte, o un predicado mal escrito). La idea de este primer laboratorio es usar Fast Downward como **verificador** del modelo y como herramienta de depuración.